# Stage 2 — Ontology-Based Semantic Enrichment

**Reads:** `outputs/lime_results.json`  
**Writes:** `outputs/ontology_results.json`

For each text, runs the classifier to get the predicted class, then maps
the top LIME features to ontology concepts and selects user-adaptive ancestors.

Each record saved:
```json
{
  "text": "...",
  "predicted_class": "Cardiovascular diseases",
  "confidence": 0.95,
  "user_category": "EXPERT",
  "ablation_mode": "normal",
  "feature_data": [
    {"feature_word": "hypertension", "lime_score": 0.42, "ancestors": [...]}
  ]
}
```

> **Tip:** Change `USER_CATEGORY` or `ABLATION_MODE` below and set `FORCE_RERUN = True` to re-run with different settings without touching Stage 1.

## 1. Imports

In [1]:
from config import (
    ABLATION_MODE,
    LIME_RESULTS_PATH,
    ONTOLOGY_RESULTS_PATH,
    USER_CATEGORY,
)
from model_loaders import load_classifier, load_ontology_model
from pipeline_helpers import (
    checkpoint_exists,
    enrich_with_ontology,
    load_checkpoint,
    predict_class,
    save_checkpoint,
)

c:\Users\vimal\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configuration

In [2]:
# Set True to re-run even if ontology_results.json already exists
FORCE_RERUN = False

# Override config values here for quick experiments
# Options: "BEGINNER" | "INTERMEDIATE" | "EXPERT"
USER_CATEGORY = USER_CATEGORY

# Options: "normal" | "full" | "one_parent" | "no_ontology"
ABLATION_MODE = ABLATION_MODE

print(f"User category : {USER_CATEGORY}")
print(f"Ablation mode : {ABLATION_MODE}")

User category : EXPERT
Ablation mode : normal


## 3. Checkpoint check

In [3]:
if checkpoint_exists(ONTOLOGY_RESULTS_PATH) and not FORCE_RERUN:
    print(f"⚠️  Checkpoint found at '{ONTOLOGY_RESULTS_PATH}'.")
    print("    Set FORCE_RERUN = True to overwrite.")
    print("    Loading existing results …")
    results = load_checkpoint(ONTOLOGY_RESULTS_PATH)
else:
    results = None
    print("No checkpoint found (or FORCE_RERUN=True). Will run ontology enrichment.")

No checkpoint found (or FORCE_RERUN=True). Will run ontology enrichment.


## 4. Load Stage 1 output

In [4]:
if results is None:
    if not LIME_RESULTS_PATH.exists():
        raise FileNotFoundError(
            f"LIME results not found at '{LIME_RESULTS_PATH}'.\n"
            "Please run 01_lime.ipynb first."
        )
    lime_data = load_checkpoint(LIME_RESULTS_PATH)
    print(f"Loaded {len(lime_data)} texts from Stage 1.")

[Checkpoint] Loaded 1 records ← 'outputs\lime_results.json'
Loaded 1 texts from Stage 1.


## 5. Load models

In [5]:
if results is None:
    _, classifier_pipeline = load_classifier()
    ontology = load_ontology_model()

Device set to use cpu


[Loader] Loading classifier from 'C:/Users/vimal/OneDrive/Documents/Uni/BTP/User-Adaptive-XAI/Models/my_medical_model' …
[Loader] Classifier ready.

[Ontology] Loading from 'C:/Users/vimal/OneDrive/Documents/Uni/BTP/User-Adaptive-XAI/Ontology/doid.owl' …
[Ontology] Loaded successfully.
[Ontology] Stats → classes: 14493, est. max depth: 15


## 6. Ontology enrichment

In [6]:
if results is None:
    results = []
    for i, item in enumerate(lime_data):
        text          = item["text"]
        lime_features = item["lime_features"]

        print(f"\nProcessing text {i + 1}/{len(lime_data)} …")

        # Predict class
        predicted_class, confidence = predict_class(text, classifier_pipeline)
        print(f"  Prediction : {predicted_class} (confidence: {confidence:.2f})")

        # Map LIME features → ontology ancestors
        feature_data = enrich_with_ontology(
            lime_features=lime_features,
            ontology=ontology,
            user_category=USER_CATEGORY,
            ablation_mode=ABLATION_MODE,
        )

        found = [f['feature_word'] for f in feature_data]
        print(f"  Ontology hits : {found} ({len(found)}/{len(lime_features[:6])} features)")

        results.append({
            "text":            text,
            "predicted_class": predicted_class,
            "confidence":      confidence,
            "user_category":   USER_CATEGORY,
            "ablation_mode":   ABLATION_MODE,
            "feature_data":    feature_data,
        })

    print("\n✅ Ontology enrichment complete.")


Processing text 1/1 …
  Prediction : Cardiovascular diseases (confidence: 0.94)
  Ontology hits : ['hypertension'] (1/6 features)

✅ Ontology enrichment complete.


## 7. Inspect results

In [7]:
for i, r in enumerate(results):
    print(f"\n── Text {i + 1} ──────────────────────────")
    print(f"Predicted class : {r['predicted_class']} ({r['confidence']})")
    for feat in r["feature_data"]:
        print(f"  {feat['feature_word']:25s} → {feat['ancestors']}")


── Text 1 ──────────────────────────
Predicted class : Cardiovascular diseases (0.9417)
  hypertension              → ['cardiovascular system disease', 'vascular disease', 'artery disease', 'hypertension']


## 8. Save checkpoint

In [8]:
save_checkpoint(results, ONTOLOGY_RESULTS_PATH)
print(f"\n➡️  Continue to notebook 03_llm.ipynb")

[Checkpoint] Saved 1 records → 'outputs\ontology_results.json'

➡️  Continue to notebook 03_llm.ipynb
